[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C34_Agent_Orchestration_Course/02_orchestration/02_orchestration.ipynb)

# 02 · 编排模式（Orchestration）

目标：用**纯标准库**从零实现四种编排模式 + 分解/合成——**pipeline → fan-out → router → supervisor-worker**，全程用 **MockLLM** 当每个 agent 的大脑、`assert` 验证，**无需 API key**。

路线：MockLLM + 分解/合成 → pipeline(带 gate) → fan-out + 合成策略 → router(分类+分派+兜底) → supervisor-worker(动态+轮数兜底) → ✏️ 练习 → 📖 答案 → 🧪 真实 orchestrator-worker 编排胶囊。

> 心智模型：**编排 = 给多个 agent 选一种接线方式**（直线/并联/分流/调度员）。能写死路径就别让模型乱接；选对拓扑比堆 agent 重要。

## 1 · MockLLM + 编排的两块基石：分解与合成

先落地 MockLLM 与两个通用原语：`decompose`（任务→子任务）、`synthesize`（子结果→最终答案）。
所有拓扑都是这两个原语 + 不同「中间连接方式」的组合。

In [ ]:
import json

class MockLLM:
    def __init__(self, rules, default='(no rule)'):
        self.rules, self.default = rules, default
        self.calls = 0
    def __call__(self, prompt):
        self.calls += 1
        text = prompt if isinstance(prompt, str) else json.dumps(prompt, ensure_ascii=False)
        for kw, resp in self.rules:
            if kw in text:
                return resp
        return self.default

def decompose(task, by='，'):
    '''最简分解：按分隔符切成子任务(真实里可让 LLM 决定)。'''
    return [s.strip() for s in task.split(by) if s.strip()]

def synthesize(task, results, sep='；'):
    '''最简合成：把子结果连起来(真实里复杂任务会再过一遍 LLM)。'''
    return sep.join(str(r) for r in results)

subs = decompose('起草文案，校对润色，翻译英文')
print('分解 ->', subs)
assert subs == ['起草文案', '校对润色', '翻译英文']
assert synthesize('t', ['A', 'B', 'C']) == 'A；B；C'
print('✅ 分解 + 合成两块基石就绪')

## 2 · 模式一：pipeline（流水线，带 gate）

把步骤串成链：上一步输出 = 下一步输入。再在步骤间加 **gate**：中间结果不合格就停 / 走补救。
适合能拆成**固定顺序**子步骤的任务（起草→校对→翻译）。最可控、最好调试。

In [ ]:
def run_pipeline(x, steps, gate=None):
    '''顺序执行 steps；每步后若 gate(中间结果) 为 False 则提前停。
       返回 {'output':最终或中断处的值, 'completed':是否全程跑完, 'stopped_at':中断步序号}。'''
    for i, step in enumerate(steps):
        x = step(x)
        if gate is not None and not gate(x):
            return {'output': x, 'completed': False, 'stopped_at': i}
    return {'output': x, 'completed': True, 'stopped_at': None}

# 文案流水线: 起草 -> 校对 -> 翻译
draft   = lambda topic: f'草稿:{topic}'
proof   = lambda d: d.replace('草稿', '已校对')
trans   = lambda d: 'EN:' + d

ok = run_pipeline('新品发布', [draft, proof, trans])
print('全程跑完 ->', ok)
assert ok['completed'] is True and ok['output'] == 'EN:已校对:新品发布'

# 带 gate: 要求中间结果长度 >= 6，否则中断(模拟『字数不达标就停』)
short_draft = lambda t: '短'           # 故意产出过短中间结果
res = run_pipeline('x', [short_draft, proof, trans], gate=lambda v: len(v) >= 6)
print('被 gate 拦下 ->', res)
assert res['completed'] is False and res['stopped_at'] == 0   # 第 0 步后就被拦
print('✅ pipeline：步骤串联 + gate 拦截不合格中间结果')

## 3 · 模式二：fan-out（并行扇出 + 合成策略）

互不依赖的子任务**同时**派 worker（顺序模拟并发），再合成。
三种合成：**拼接**(互补块)、**归约**(同质，如求和)、**再合成**(过 LLM 整合)。失败**尽力合成**。

In [ ]:
def fan_out(subtasks, worker):
    '''对每个子任务派一个 worker；失败隔离成 failed 结果(结果数不变)。'''
    out = []
    for i, st in enumerate(subtasks):
        try:
            out.append({'id': i, 'task': st, 'status': 'ok', 'result': worker(st)})
        except Exception as e:
            out.append({'id': i, 'task': st, 'status': 'failed', 'error': str(e)})
    return out

def synth_concat(results):
    return '；'.join(r['result'] for r in results if r['status'] == 'ok')
def synth_reduce(results, fn):
    return fn([r['result'] for r in results if r['status'] == 'ok'])

# sectioning: 分析文档三个章节(互补) -> 拼接
analyze = lambda sec: f'{sec}要点'
res = fan_out(['第一章', '第二章', '第三章'], analyze)
assert len(res) == 3 and all(r['status'] == 'ok' for r in res)
assert synth_concat(res) == '第一章要点；第二章要点；第三章要点'

# 归约: 各 worker 数字数 -> 求和
res2 = fan_out(['abc', 'de', 'f'], lambda s: len(s))
assert synth_reduce(res2, sum) == 6

# 尽力合成: 一个 worker 失败，仍用成功的合成
def flaky(s):
    if s == 'bad': raise ValueError('坏块')
    return s + '!'
res3 = fan_out(['a', 'bad', 'c'], flaky)
assert len(res3) == 3 and synth_concat(res3) == 'a!；c!'   # 跳过失败块
print('拼接:', synth_concat(res), '| 归约:', synth_reduce(res2, sum), '| 尽力:', synth_concat(res3))
print('✅ fan-out：并行扇出 + 三种合成策略 + 失败尽力合成')

## 4 · 模式三：router（路由：分类 → 分派 → 兜底）

输入种类杂时，先**分类**再分派给**专精**下游。类别要互斥穷尽，且有 **default 兜底**接住未匹配的。
router 与工具分发、权限策略同构——都是『按 key 查表分派』。

In [ ]:
def classify(text):
    '''规则分类器(真实里可换轻量 LLM)：返回类别标签。'''
    if '退款' in text or '退货' in text:   return 'refund'
    if '物流' in text or '快递' in text:   return 'logistics'
    if '报错' in text or '故障' in text:   return 'tech'
    return 'default'

HANDLERS = {
    'refund':    lambda t: '【退款专员】已为您发起退款流程',
    'logistics': lambda t: '【物流专员】您的包裹预计明天送达',
    'tech':      lambda t: '【技术支持】请尝试重启后重试',
    'default':   lambda t: '【人工客服】已为您转接人工',
}

def route(text, classify_fn, handlers):
    label = classify_fn(text)
    handler = handlers.get(label, handlers['default'])   # 兜底
    return {'label': label, 'reply': handler(text)}

for q in ['我要退款', '查一下物流', '系统报错了', '随便聊聊']:
    r = route(q, classify, HANDLERS)
    print(f'{q:8s} -> [{r["label"]:9s}] {r["reply"]}')
assert route('我要退款', classify, HANDLERS)['label'] == 'refund'
assert route('随便聊聊', classify, HANDLERS)['label'] == 'default'   # 兜底
assert '退款' in route('我要退款', classify, HANDLERS)['reply']
print('✅ router：分类 -> 分派给专精下游 -> default 兜底(无输入无路可走)')

## 5 · 模式四：supervisor-worker（动态编排 + 轮数兜底）

路径无法预先定时，让一个 **supervisor** 在循环里动态决定**派谁/几个 worker**、看结果后是否继续、何时收尾。
最灵活也最贵最难控——必须配 **轮数上限** 兜底防死循环。

In [ ]:
def supervisor_orchestrate(goal, supervisor_fn, worker_fn, max_rounds=5):
    '''supervisor_fn(goal, history) -> {'action':'dispatch'/'done', 'tasks':[...], 'answer':...}。
       每轮: supervisor 决策 -> 若 dispatch 则派 worker 收结果存 history -> 直到 done 或超轮数。'''
    history = []
    for rnd in range(max_rounds):
        decision = supervisor_fn(goal, history)
        if decision['action'] == 'done':
            return {'answer': decision['answer'], 'rounds': rnd, 'history': history}
        # dispatch: 派出本轮 worker(可多个), 收结果
        round_results = [worker_fn(t) for t in decision['tasks']]
        history.append({'round': rnd, 'tasks': decision['tasks'], 'results': round_results})
    return {'answer': '[超出最大轮数]', 'rounds': max_rounds, 'history': history}

# 一个确定性 supervisor: 第一轮收集事实，第二轮基于事实给结论，然后 done
def supervisor(goal, history):
    if len(history) == 0:
        return {'action': 'dispatch', 'tasks': ['查事实A', '查事实B']}   # 动态决定派 2 个
    if len(history) == 1:
        facts = history[0]['results']
        return {'action': 'dispatch', 'tasks': [f'基于{facts}做结论']}    # 依据上轮结果再派
    return {'action': 'done', 'answer': '最终结论: ' + str(history[-1]['results'])}

worker = lambda t: t + '✓'
out = supervisor_orchestrate('完成调研', supervisor, worker, max_rounds=5)
print('用了轮数:', out['rounds'])
print('最终答案:', out['answer'])
assert out['rounds'] == 2                         # 两轮 dispatch 后 done
assert '最终结论' in out['answer']
assert len(out['history']) == 2
# 轮数兜底: 一个永不 done 的 supervisor 必须被 max_rounds 截停
endless = lambda g, h: {'action': 'dispatch', 'tasks': ['x']}
out2 = supervisor_orchestrate('g', endless, worker, max_rounds=3)
assert out2['rounds'] == 3 and out2['answer'] == '[超出最大轮数]'
print('✅ supervisor-worker：动态派 worker、依据进展再决策；轮数兜底防死循环')

## 6 · 组合：一个端到端 orchestrator（router → 各模式）

真实编排是模式的**组合**。这里：先 router 分类，研究类走 fan-out，文案类走 pipeline，其它兜底。
验证不同输入走不同拓扑、各自产出正确结果。

In [ ]:
orch_llm = MockLLM(rules=[('A公司', 'A +10%'), ('B公司', 'B +12%')])

def top_classify(task):
    if '调研' in task or '对比' in task:  return 'research'
    if '文案' in task or '写' in task:    return 'copywriting'
    return 'default'

def handle_research(task):
    subs = ['调研 A公司', '调研 B公司']            # 分解
    res = fan_out(subs, orch_llm)                  # fan-out
    return {'pattern': 'fan-out', 'answer': synth_concat(res)}

def handle_copy(task):
    out = run_pipeline(task, [draft, proof, trans])  # pipeline
    return {'pattern': 'pipeline', 'answer': out['output']}

def top_orchestrate(task):
    label = top_classify(task)
    if label == 'research':     return {'label': label, **handle_research(task)}
    if label == 'copywriting':  return {'label': label, **handle_copy(task)}
    return {'label': 'default', 'pattern': 'single', 'answer': '直接回答: ' + task}

r1 = top_orchestrate('对比 A、B 公司')
r2 = top_orchestrate('写一段产品文案')
r3 = top_orchestrate('今天几号')
print(r1['label'], '->', r1['pattern'], '->', r1['answer'])
print(r2['label'], '->', r2['pattern'], '->', r2['answer'])
print(r3['label'], '->', r3['pattern'])
assert r1['pattern'] == 'fan-out' and 'A +10%' in r1['answer']
assert r2['pattern'] == 'pipeline' and r2['answer'].startswith('EN:')
assert r3['pattern'] == 'single'
print('✅ 组合编排：router 顶层分流 -> 研究走 fan-out、文案走 pipeline、其余兜底')

---
## ✏️ 练习 1：依赖图（DAG）拓扑排序调度

比『独立子任务列表』更通用的是**依赖图**：有依赖的须等前驱完成。

实现 `topo_order(deps)`：`deps` 是 `{任务: [它依赖的任务...]}`；返回一个**合法执行顺序**（任一任务排在它依赖的任务之后）。若有环则 `raise ValueError('cycle')`。(用 Kahn 算法：反复取出入度为 0 的任务。)

In [ ]:
def topo_order(deps):
    # TODO: Kahn 算法
    #  1) 算每个任务的入度(它依赖几个)；所有出现的任务都要计入
    #  2) 反复取入度为 0 的任务加入结果、并把『依赖它的任务』入度-1
    #  3) 若结果数 != 任务总数 -> 有环 -> raise ValueError('cycle')
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# 退款流程: 退款依赖查订单, 查订单依赖核身份
deps = {'核身份': [], '查订单': ['核身份'], '退款': ['查订单']}
order = topo_order(deps)
assert order.index('核身份') < order.index('查订单') < order.index('退款')
# 并行机会: A、B 都只依赖 root，可在 root 之后任意序
deps2 = {'root': [], 'A': ['root'], 'B': ['root'], 'merge': ['A', 'B']}
o2 = topo_order(deps2)
assert o2[0] == 'root' and o2[-1] == 'merge'
# 有环要报错
try:
    topo_order({'x': ['y'], 'y': ['x']}); raised = False
except ValueError:
    raised = True
assert raised
print('合法执行序:', order)
print('✅ 练习 1 通过：依赖图拓扑排序，尊重依赖、检测环')

## ✏️ 练习 2：voting 式 fan-out（多数共识）

fan-out 的另一形态是 **voting**：同一任务跑多次取共识，用冗余换可靠。

实现 `vote(task, voters)`：`voters` 是一组评判函数，各对 `task` 返回一个标签；返回**得票最多**的标签（平票取任一最高即可）和票数分布。

In [ ]:
from collections import Counter
def vote(task, voters):
    # TODO: votes = [v(task) for v in voters]
    #   tally = Counter(votes)；winner = 票数最多的标签
    #   返回 {'winner': winner, 'tally': dict(tally)}
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 三个 agent 判断这段代码有没有漏洞：两个说 vuln、一个说 safe
voters = [lambda t: 'vuln', lambda t: 'vuln', lambda t: 'safe']
out = vote('def f(x): eval(x)', voters)
assert out['winner'] == 'vuln'                 # 多数说有漏洞
assert out['tally'] == {'vuln': 2, 'safe': 1}
print('共识:', out['winner'], '| 票数:', out['tally'])
print('✅ 练习 2 通过：voting 取多数共识，用冗余提升可靠')

## ✏️ 练习 3：evaluator-optimizer（评估-优化回路）

给编排加自检：一个 producer 产结果，一个 evaluator 评分，不达标就带反馈重做，循环到达标或超次数。

实现 `eval_optimize(task, produce, evaluate, threshold, max_iters)`：`produce(task, feedback)` 产出结果，`evaluate(result)` 返回 `(score, feedback)`；循环直到 `score >= threshold` 或达 `max_iters`；返回 `{'result', 'score', 'iters', 'passed'}`。

In [ ]:
def eval_optimize(task, produce, evaluate, threshold, max_iters=5):
    # TODO: feedback=None
    #   循环最多 max_iters 次: result=produce(task,feedback); score,feedback=evaluate(result)
    #     若 score>=threshold -> 返回 {'result','score','iters':本次序号+1,'passed':True}
    #   超次数 -> 返回最后一次 {... 'passed':False}
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# producer: 每轮把质量+1(用反馈里的当前分推进)；evaluator: 分=结果长度
def produce(task, feedback):
    base = 0 if feedback is None else feedback['len']
    return '好' * (base + 1)                    # 每轮变长一点
def evaluate(result):
    score = len(result)
    return score, {'len': score}
out = eval_optimize('写句子', produce, evaluate, threshold=3, max_iters=5)
assert out['passed'] is True and out['score'] >= 3 and out['iters'] == 3
# 永远不达标 -> 超次数, passed False
out2 = eval_optimize('x', lambda t, f: 'a', lambda r: (0, None), threshold=10, max_iters=4)
assert out2['passed'] is False and out2['iters'] == 4
print('达标用了', out['iters'], '轮; 结果=', out['result'])
print('✅ 练习 3 通过：评估-优化回路，迭代到达标或超次数')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案（Kahn 拓扑排序）
def topo_order(deps):
    nodes = set(deps) | {d for ds in deps.values() for d in ds}
    indeg = {n: 0 for n in nodes}
    for n, ds in deps.items():
        indeg[n] = len(ds)
    # 谁依赖谁的反向边
    dependents = {n: [] for n in nodes}
    for n, ds in deps.items():
        for d in ds:
            dependents[d].append(n)
    ready = sorted([n for n in nodes if indeg[n] == 0])
    order = []
    while ready:
        n = ready.pop(0)
        order.append(n)
        for m in dependents[n]:
            indeg[m] -= 1
            if indeg[m] == 0:
                ready.append(m)
        ready.sort()
    if len(order) != len(nodes):
        raise ValueError('cycle')
    return order

In [ ]:
# 练习 2 参考答案
from collections import Counter
def vote(task, voters):
    votes = [v(task) for v in voters]
    tally = Counter(votes)
    winner = tally.most_common(1)[0][0]
    return {'winner': winner, 'tally': dict(tally)}

In [ ]:
# 练习 3 参考答案
def eval_optimize(task, produce, evaluate, threshold, max_iters=5):
    feedback = None
    result, score = None, None
    for i in range(max_iters):
        result = produce(task, feedback)
        score, feedback = evaluate(result)
        if score >= threshold:
            return {'result': result, 'score': score, 'iters': i + 1, 'passed': True}
    return {'result': result, 'score': score, 'iters': max_iters, 'passed': False}

---
## 🧪 真实数据胶囊：Building effective agents 的五种模式映射

Anthropic《Building effective agents》给出五种可组合的 workflow 模式。下面把它们与本课实现一一对应、并跑一个**贴近真实**的「研究报告生成」组合编排（router → 分解 → fan-out → evaluator → 合成）。

> 形状对照：真实里每个『步骤 / worker』是一次 `messages.create(model='claude-opus-4-8', ...)`；本课用 MockLLM 模拟。

In [ ]:
# 五种模式 -> 本课实现 的对照表
PATTERN_MAP = {
    'prompt chaining':     'run_pipeline',       # 流水线
    'routing':             'route',              # 路由
    'parallelization':     'fan_out + synth',    # 并行/扇出
    'orchestrator-workers':'supervisor_orchestrate',  # 主从
    'evaluator-optimizer': 'eval_optimize',      # 评估-优化(练习3)
}
for k, v in PATTERN_MAP.items():
    print(f'{k:22s} -> {v}')
assert len(PATTERN_MAP) == 5

# 贴近真实的组合: 生成一份『三公司对比报告』
report_llm = MockLLM(rules=[('A公司', 'A: 营收+10%'), ('B公司', 'B: 营收+12%'),
                            ('C公司', 'C: 营收-3%')])
def gen_report(task):
    if top_classify_v2(task) != 'research':
        return {'pattern': 'single', 'report': '（非研究任务）'}
    subs = ['调研 A公司', '调研 B公司', '调研 C公司']      # 分解
    res = fan_out(subs, report_llm)                       # parallelization
    body = synth_concat(res)                             # 合成
    return {'pattern': 'router->fan-out->synth', 'report': '对比报告：' + body,
            'n_workers': len(res)}
def top_classify_v2(task):
    return 'research' if ('对比' in task or '调研' in task) else 'default'

out = gen_report('对比 A、B、C 三家公司')
print(out['pattern'], '| workers =', out['n_workers'])
print(out['report'])
assert out['n_workers'] == 3 and 'A: 营收+10%' in out['report']
print('✅ 复现 Building effective agents 的组合编排：路由→分解→扇出→合成')

**🧪 胶囊练习**：实现 `pick_pattern(task)`：给定任务描述，按简单规则返回推荐的编排模式名——含『步骤/先后/再』→`'prompt chaining'`；含『对比/分别/并行』→`'parallelization'`；含『分类/不同类型』→`'routing'`；否则 `'single LLM call'`。（这就是模块讲的『动手前先选对拓扑』的最小版。）

In [ ]:
def pick_pattern(task):
    # TODO: 按关键词返回推荐模式名(见题面)
    raise NotImplementedError

In [ ]:
# 自测
assert pick_pattern('先起草再校对') == 'prompt chaining'
assert pick_pattern('分别分析三个章节') == 'parallelization'
assert pick_pattern('按不同类型分类处理工单') == 'routing'
assert pick_pattern('今天几号') == 'single LLM call'
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def pick_pattern(task):
    if any(w in task for w in ['步骤', '先', '再', '然后']):       return 'prompt chaining'
    if any(w in task for w in ['对比', '分别', '并行', '各个']):   return 'parallelization'
    if any(w in task for w in ['分类', '不同类型', '不同类别']):   return 'routing'
    return 'single LLM call'

---
## 🔧 旁注：对应的真实 Claude 编排

本课用 MockLLM 跑通的四种编排，换成真实 Claude 只是把每个『步骤 / worker / 分类器 / supervisor』换成一次 `messages.create`（伪代码，**本环境不跑、需 API key；无 key 自动回退 MockLLM**）：

```python
import anthropic
client = anthropic.Anthropic()

def llm_step(prompt):                                  # 把 MockLLM 那一行换成这个
    r = client.messages.create(model='claude-opus-4-8', max_tokens=1024,
                               messages=[{'role':'user','content':prompt}])
    return ''.join(b.text for b in r.content if b.type=='text')

# pipeline: steps = [起草提示, 校对提示, 翻译提示] 各 llm_step
# fan-out : 用线程池对每个子任务并发 llm_step
# router  : classify 用 llm_step(分类提示) 或一个小模型
# supervisor: supervisor_fn 内部是一次 llm_step, 让模型输出 JSON 决策
```

对应关系：`run_pipeline` / `fan_out` / `route` / `supervisor_orchestrate` 的**编排控制流一行不改**，只把节点处的 `MockLLM(...)` 换成 `llm_step(...)`。这就是「scaffold 可迁移」——你练的是编排，不是模型。

### 小结
- 编排 = 给多个 agent 选**接线方式**；两块基石是**分解**(入口)与**合成**(出口)。
- **pipeline**：步骤串成链 + gate 拦截；最可控，适合固定顺序步骤。
- **fan-out**：互不依赖子任务并行 + 拼接/归约/再合成；失败尽力合成。
- **router**：分类→分派给专精下游 + default 兜底；关注点分离、可插拔。
- **supervisor-worker**：supervisor 动态决策派 worker + 轮数兜底；最灵活也最贵最难控。
- 四模式是一条**可控↔灵活**光谱；选能解决任务的**最靠左**那个，能写死就别让模型乱接。
- 真实编排是模式的**嵌套组合**；复杂编排**天然需要可观测**（→模块 04）。

下一站：**模块 03 · 权限与沙箱** —— 给这套编排装上权限边界，让每个 agent 不能乱来。